# 체육시설 안전점검 이미지 분류 - 데이터 탐색 (EDA)
본 노트북에서는 타깃 데이터(체육시설 결함 265장)와 보조 데이터(AI-Hub 일반 시설물 약 2만장)의 분포, 특성, 그리고 증강(Augmentation) 가능성을 탐색합니다.

In [ ]:
import os
import glob
import cv2
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from PIL import Image
import albumentations as A

# 플롯 스타일 설정
plt.style.use('ggplot')
%matplotlib inline

# 데이터 경로 설정 (상황에 맞게 수정하세요)
DATA_DIR = "./data/raw"
# 예시: data/raw/gym/ 폴더와 data/raw/ai_hub/ 폴더가 있다고 가정
gym_images = glob.glob(os.path.join(DATA_DIR, "gym", "*.jpg"))
ai_hub_images = glob.glob(os.path.join(DATA_DIR, "ai_hub", "*.jpg"))

print(f"체육시설 타깃 이미지 수: {len(gym_images)}")
print(f"AI-Hub 보조 이미지 수: {len(ai_hub_images)}")

## 1. 도메인 차이 육안 확인 (Gym vs AI-Hub)
우리가 우려했던 **도메인 차이 리스크**를 눈으로 확인합니다. 체육시설의 배경(우레탄 바닥 등)과 일반 건물의 배경이 어떻게 다른지 비교합니다.

In [ ]:
def show_images(image_paths, title):
    plt.figure(figsize=(15, 5))
    for i, path in enumerate(image_paths[:5]): # 5장만 샘플로 출력
        if not os.path.exists(path): continue
        
        img = cv2.imread(path)
        img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
        
        plt.subplot(1, 5, i+1)
        plt.imshow(img)
        plt.axis('off')
        plt.title(f"Sample {i+1}")
    plt.suptitle(title, fontsize=16)
    plt.show()

# 이미지가 폴더에 들어있다고 가정하고 출력 (샘플용 더미 데이터가 있다면 실행됨)
if gym_images and ai_hub_images:
    show_images(gym_images, "Gym Facility Images (Target Domain)")
    show_images(ai_hub_images, "AI-Hub General Building Images (Source Domain)")

## 2. 클래스 불균형 (Long-tail) 확인
총 22개의 클래스가 존재합니다. 정상 데이터와 특정 위험 데이터의 비율이 얼마나 차이 나는지 분포를 확인합니다.

In [ ]:
# 실제로는 메타데이터 CSV 파일을 pandas로 읽어와야 합니다.
# 예시: df = pd.read_csv('data/labels.csv')
# 여기서는 가상의 분포를 생성하여 불균형을 확인합니다.

classes = [f"Class_{i}" for i in range(22)]
# 가상의 클래스 분포 (정상 클래스가 압도적으로 많은 Long-tail 흉내)
counts = [12000, 3000, 1500, 800, 500] + [np.random.randint(50, 200) for _ in range(17)]

plt.figure(figsize=(12, 6))
plt.bar(classes, counts, color='royalblue')
plt.xticks(rotation=45, ha='right')
plt.title("Class Distribution (Long-tail check)")
plt.ylabel("Number of Images")
plt.show()

print("👉 분석 결과: 예상대로 소수 클래스의 데이터가 매우 부족하므로 WeightedRandomSampler 적용이 필수적임.")

## 3. 병목 가설 검증: 원본 이미지 해상도 분석
워크시트 Part D에서 세웠던 "고해상도 이미지 리사이징으로 인한 I/O 병목" 가설을 위해, 원본 이미지들의 해상도를 파악합니다.

In [ ]:
# 샘플 이미지 100장의 크기만 읽어서 분포 확인 (시간 절약을 위해 100장만)
widths, heights = [], []
for path in ai_hub_images[:100]:
    with Image.open(path) as img:
        widths.append(img.width)
        heights.append(img.height)

if widths:
    plt.figure(figsize=(8, 6))
    plt.scatter(widths, heights, alpha=0.5)
    plt.axvline(x=224, color='r', linestyle='--', label='ViT Input Size (224)')
    plt.axhline(y=224, color='r', linestyle='--')
    plt.title("Original Image Resolutions")
    plt.xlabel("Width")
    plt.ylabel("Height")
    plt.legend()
    plt.show()

    print(f"평균 해상도: {np.mean(widths):.0f} x {np.mean(heights):.0f}")
    print("👉 분석 결과: 모델 입력(224x224) 대비 원본 해상도가 너무 커서 전처리 병목이 예상됨. 학습 전 사전 리사이징(Pre-resizing) 고려 필요.")

## 4. 데이터 증강(Augmentation) 시뮬레이션
부족한 체육시설 데이터 265장에 Albumentations를 활용한 강한 증강이 잘 먹히는지 시뮬레이션합니다.

In [ ]:
# Albumentations 파이프라인 정의 (워크시트에서 기획한 강력한 증강)
transform = A.Compose([
    A.HorizontalFlip(p=0.5),
    A.RandomBrightnessContrast(brightness_limit=0.2, contrast_limit=0.2, p=0.8),
    A.ShiftScaleRotate(shift_limit=0.06, scale_limit=0.1, rotate_limit=15, p=0.5),
    A.GaussNoise(var_limit=(10.0, 50.0), p=0.5) # 실내 저화질 폰카메라 노이즈 모사
])

if gym_images:
    sample_img = cv2.imread(gym_images[0])
    sample_img = cv2.cvtColor(sample_img, cv2.COLOR_BGR2RGB)
    
    plt.figure(figsize=(15, 5))
    plt.subplot(1, 4, 1)
    plt.imshow(sample_img)
    plt.title("Original")
    plt.axis('off')
    
    # 동일한 이미지에 각기 다른 증강 3번 적용
    for i in range(3):
        augmented = transform(image=sample_img)['image']
        plt.subplot(1, 4, i+2)
        plt.imshow(augmented)
        plt.title(f"Augmented {i+1}")
        plt.axis('off')
    plt.show()

## 5. 베이스라인: OpenCV 엣지 검출 (규칙 기반)
딥러닝 적용 전, 베이스라인으로 설정했던 Canny Edge Detection이 균열(Crack)을 어느 정도 찾아내는지 테스트합니다.

In [ ]:
if gym_images:
    sample_crack = cv2.imread(gym_images[0], cv2.IMREAD_GRAYSCALE)
    
    # 가우시안 블러로 자잘한 노이즈 제거 후 Canny Edge 검출
    blurred = cv2.GaussianBlur(sample_crack, (5, 5), 0)
    edges = cv2.Canny(blurred, 50, 150)
    
    plt.figure(figsize=(10, 5))
    plt.subplot(1, 2, 1)
    plt.imshow(sample_crack, cmap='gray')
    plt.title("Original Gray Image")
    plt.axis('off')
    
    plt.subplot(1, 2, 2)
    plt.imshow(edges, cmap='gray')
    plt.title("Canny Edge Detection")
    plt.axis('off')
    plt.show()
    
    print("👉 분석 결과: 선명한 균열은 잡히지만, 벽면의 무늬나 우레탄 바닥의 패턴도 엣지로 인식되는 한계가 있음. -> 규칙 기반 한계 증명 (ML 딥러닝 도입의 타당성 확보)")